# 🤖 AIエージェント with llama-cpp-python + Phi-3.5

| 項目 | 内容 |
|------|------|
| **推論エンジン** | llama-cpp-python (pip installのみ・ビルド不要) |
| **LLM** | Phi-3.5-mini-instruct (Microsoft) |
| **モデルサイズ** | 約2.2GB (Q4_K_M量子化) |
| **登録** | 不要 |
| **GPU** | T4推奨 |

## Step 1: インストール

In [ ]:
# CUDA対応のllama-cpp-pythonをインストール
import subprocess
result = subprocess.run(
    ['pip', 'install', 'llama-cpp-python',
     '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121',
     '--upgrade', '-q'],
    env={**__import__('os').environ, 'CMAKE_ARGS': '-DGGML_CUDA=on'},
    capture_output=True, text=True
)
print('✅ インストール完了')

## Step 2: GPU確認

In [ ]:
import torch
from llama_cpp import Llama

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({vram:.1f} GB VRAM)')
else:
    print('⚠️  GPU未検出 → ランタイム > T4 GPU を選択してください')

print('✅ llama-cpp-python インポート成功')

## Step 3: Phi-3.5 モデルのダウンロード（登録不要）

In [ ]:
%%bash
MODEL_PATH="/content/phi-3.5-mini-q4.gguf"
MODEL_URL="https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf"

if [ -f "$MODEL_PATH" ]; then
  echo "✅ モデルは既にダウンロード済みです"
else
  echo "📥 Phi-3.5-mini をダウンロード中... (~2.2GB)"
  wget -q --show-progress -O "$MODEL_PATH" "$MODEL_URL"
  echo "✅ ダウンロード完了"
fi
ls -lh "$MODEL_PATH"

## Step 4: モデルのロード

In [ ]:
from llama_cpp import Llama

print('📥 Phi-3.5 をロード中...')

llm = Llama(
    model_path='/content/phi-3.5-mini-q4.gguf',
    n_gpu_layers=35,      # T4の場合は35前後。全層GPUなら -1
    n_ctx=4096,
    n_batch=512,
    verbose=False,
    chat_format='chatml',
)

print('✅ モデルロード完了')

# 動作確認
resp = llm.create_chat_completion(
    messages=[{'role': 'user', 'content': '「元気です」を英語で1語だけ答えてください。'}],
    max_tokens=10,
    temperature=0.1,
)
print(f'✅ 動作確認: {resp["choices"][0]["message"]["content"].strip()}')

## Step 5: LLM呼び出し関数

In [ ]:
def llm_chat(messages: list, max_tokens: int = 512, temperature: float = 0.3) -> str:
    """llama-cpp-python でチャット補完"""
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=['<|end|>', '<|user|>'],
    )
    return resp['choices'][0]['message']['content'].strip()

print('✅ llm_chat 関数定義完了')

## Step 6: ツール定義

In [ ]:
import re, json, math

def calculator(expression: str) -> str:
    try:
        clean = expression.replace('^', '**').replace('√', 'math.sqrt')
        result = eval(clean, {'__builtins__': {}, 'math': math})
        return f'{expression} = {result}'
    except Exception as e:
        return f'計算エラー: {e}'

def search_knowledge(query: str) -> str:
    kb = {
        'python':    'Pythonは1991年にGuido van Rossumが開発した高水準言語。読みやすい文法と豊富なライブラリが特徴。',
        '機械学習':  '機械学習はデータからパターンを自動的に学習するAI技術。教師あり・教師なし・強化学習の3種がある。',
        'phi':       'Phi-3.5はMicrosoftが開発した3.8Bパラメータの小型高性能LLM。MITライセンスで登録不要・商用利用可能。',
        'llama.cpp': 'llama.cppはC++製の高速LLM推論エンジン。GGUF形式のモデルをGPU/CPUで効率的に動作させる。',
        '東京':      '東京は日本の首都。人口約1,400万人（都区部）。世界最大級の都市圏の一つ。',
        '富士山':    '富士山は日本最高峰（3,776m）。静岡・山梨両県にまたがる活火山。2013年に世界文化遺産登録。',
    }
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return f'「{query}」に関する情報は見つかりませんでした。'

def data_analyzer(data_str: str) -> str:
    try:
        nums = [float(x) for x in re.findall(r'[-+]?\d*\.?\d+', data_str)]
        if not nums:
            return 'エラー: 数値が見つかりません'
        n = len(nums)
        mean = sum(nums) / n
        s = sorted(nums)
        median = s[n//2] if n % 2 else (s[n//2-1]+s[n//2])/2
        std = math.sqrt(sum((x-mean)**2 for x in nums)/n)
        return (f'件数:{n}  最小:{min(nums)}  最大:{max(nums)}\n'
                f'平均:{mean:.2f}  中央値:{median}  標準偏差:{std:.2f}')
    except Exception as e:
        return f'分析エラー: {e}'

def text_summarizer(text: str) -> str:
    if len(text) < 20:
        return f'（要約スキップ）{text}'
    return llm_chat([
        {'role': 'system', 'content': 'Summarize the following text in 2-3 Japanese sentences. Stay strictly on topic.'},
        {'role': 'user',   'content': text},
    ], max_tokens=200)

def web_search(query: str) -> str:
    return (f'「{query}」の検索結果（モック）:\n'
            f'1. {query}の最新動向 — example.com\n'
            f'2. {query}入門ガイド — guide.example.com\n'
            '※本番はDuckDuckGo Search等のAPIに差し替えてください')

TOOLS = {
    'calculator':       {'fn': calculator,       'desc': '数式を計算。例: calculator("3 * (10 + 5)")'},
    'search_knowledge': {'fn': search_knowledge,  'desc': '知識を検索。例: search_knowledge("Phi")'},
    'data_analyzer':    {'fn': data_analyzer,     'desc': '数値データを統計分析。例: data_analyzer("10,20,30")'},
    'text_summarizer':  {'fn': text_summarizer,   'desc': 'テキストを要約。例: text_summarizer("長い文章")'},
    'web_search':       {'fn': web_search,        'desc': 'Webを検索。例: web_search("最新AIニュース")'},
}
TOOL_DESCS = '\n'.join(f'- {n}: {v["desc"]}' for n, v in TOOLS.items())
print('✅ ツール定義完了\n' + TOOL_DESCS)

## Step 7: マルチステップ AIエージェント

In [ ]:
PLAN_SYSTEM = f"""You are a task planning AI. Break down the user's task into 2-4 steps.
Output ONLY a JSON array. No explanation, no markdown, no code blocks.

Available tools:
{TOOL_DESCS}

Rules:
- Use data_analyzer for statistics (mean, max, min). Do NOT use calculator for stats.
- calculator is only for simple arithmetic like "2.2 * 10".

Output format (JSON array only):
[{{"step":1,"action":"説明","tool":"tool_name_or_none","input":"input string"}}, ...]"""

SUMMARY_SYSTEM = """あなたは結果をまとめるAIです。
ユーザーの質問と各ステップの実行結果をもとに、
分かりやすい日本語で最終回答をまとめてください。"""


def plan_task(user_input: str) -> list:
    response = llm_chat([
        {'role': 'system', 'content': PLAN_SYSTEM},
        {'role': 'user',   'content': f'タスク: {user_input}'},
    ], max_tokens=400)
    try:
        m = re.search(r'\[.*?\]', response, re.DOTALL)
        steps = json.loads(m.group()) if m else []
    except:
        steps = []

    if not steps:
        steps = [{'step':1,'action':user_input,'tool':'search_knowledge','input':user_input}]

    # inputが空のステップにはuser_inputをそのまま入れる
    for s in steps:
        if not s.get('input') or s['input'].strip() == '':
            s['input'] = user_input

    return steps


def execute_step(step: dict) -> str:
    tool = step.get('tool', 'none')
    inp  = step.get('input', '')

    if tool == 'calculator':
        formula = re.search(r'[\d\s\+\-\*\/\.\(\)\^]+', inp)
        if formula and any(c.isdigit() for c in formula.group()):
            inp = formula.group().strip()
        else:
            inp = llm_chat([
                {'role': 'system', 'content': 'Output a single math expression only. No text, no units, just numbers and operators like: 2.2 * 10'},
                {'role': 'user',   'content': f'Write a math expression for: {inp}'},
            ], max_tokens=30, temperature=0.1).strip()
            inp = re.sub(r'[^0-9\+\-\*\/\.\(\)\s]', '', inp).strip()

    if tool and tool != 'none' and tool in TOOLS:
        return TOOLS[tool]['fn'](inp)
    return f'（処理済み）{step.get("action","")}'


def summarize(user_input: str, results: list) -> str:
    steps_text = '\n'.join(
        f'Step{r["step"]}: {r["action"]}\n結果: {r["result"]}' for r in results
    )
    return llm_chat([
        {'role': 'system', 'content': SUMMARY_SYSTEM},
        {'role': 'user',   'content': f'質問: {user_input}\n\n実行結果:\n{steps_text}'},
    ], max_tokens=512)


def run_agent(user_input: str) -> dict:
    SEP = '─' * 55
    print(f'\n{SEP}')
    print(f'🤖 AIエージェント  |  Phi-3.5 + llama-cpp-python')
    print(f'📋 タスク: {user_input}')
    print(SEP)

    print('\n📌 Phase 1: タスク分解中...')
    steps = plan_task(user_input)
    if not steps:
        print('  ⚠️  プランの生成に失敗しました')
        return {}
    for s in steps:
        tag = f' [🔧 {s.get("tool")}]' if s.get('tool') not in (None, 'none') else ''
        print(f'  Step {s.get("step","-")}: {s.get("action","")}{tag}')

    print('\n⚙️  Phase 2: ステップ実行中...')
    results = []
    for step in steps:
        print(f'\n  ▶ Step {step.get("step","-")}: {step.get("action","")}')
        result = execute_step(step)
        preview = result[:120] + '...' if len(result) > 120 else result
        print(f'  ✓ {preview}')
        results.append({'step': step.get('step','-'), 'action': step.get('action',''), 'result': result})

    print('\n📝 Phase 3: 最終回答を生成中...')
    final = summarize(user_input, results)

    print(f'\n{SEP}')
    print('✅ 最終回答')
    print(SEP)
    print(final)
    print(SEP)
    return {'steps': steps, 'results': results, 'final_answer': final}


print('✅ エージェント定義完了')

## Step 8: 実行してみよう！

In [ ]:
# 例1: 知識検索 + 計算
run_agent("Phi-3.5とllama.cppについて調べて、4bit量子化で2.2GBのモデルを10個ロードすると何GB必要か計算して")

In [ ]:
# 例2: データ分析
run_agent("次のテストスコアを分析して平均点と最高点を教えて: 72, 85, 91, 68, 78, 95, 83, 77, 88, 64")

In [ ]:
# 例3: 自由入力 ← ここを変えて試してください
run_agent("機械学習について調べて、初心者が学ぶべきステップを教えて")

## Step 9: インタラクティブUI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

task_box = widgets.Textarea(
    value='Phi-3.5について調べて、Colabで使うメリットをまとめて',
    description='タスク:',
    layout=widgets.Layout(width='85%', height='80px'),
)
run_btn = widgets.Button(
    description='🤖 エージェント実行',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='38px'),
)
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        run_agent(task_box.value)

run_btn.on_click(on_click)
display(task_box, run_btn, out)

---
## 📚 カスタマイズガイド

### 別モデルへの切り替え（すべて登録不要）
```bash
# Qwen2.5-3B（日本語が強い）
wget https://huggingface.co/bartowski/Qwen2.5-3B-Instruct-GGUF/resolve/main/Qwen2.5-3B-Instruct-Q4_K_M.gguf

# Gemma-3-1B（超軽量）
wget https://huggingface.co/bartowski/gemma-3-1b-it-GGUF/resolve/main/gemma-3-1b-it-Q4_K_M.gguf
```
```python
# ロード時にモデルパスとchat_formatを変更するだけ
llm = Llama(model_path='/content/Qwen2.5-3B-Instruct-Q4_K_M.gguf',
            chat_format='chatml', n_gpu_layers=35, n_ctx=4096, verbose=False)
```

### GPU層数の調整
```python
n_gpu_layers=-1  # 全層GPU（VRAMが十分な場合・最速）
n_gpu_layers=0   # CPU推論（GPUなし環境）
```

### 実Web検索への差し替え
```python
!pip install duckduckgo-search
from duckduckgo_search import DDGS

def web_search(query: str) -> str:
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
        return '\n'.join(r['body'] for r in results)
TOOLS['web_search']['fn'] = web_search
```

### 独自ツールの追加
```python
def my_tool(input: str) -> str:
    # 処理
    return result

TOOLS['my_tool'] = {'fn': my_tool, 'desc': 'ツールの説明'}
```